# ⚽ Mission 18 — Build the AI Soccer Agent (Capstone Product)
### From Python Functions to a Real Multi-Page AI Agent Product

Welcome to **Mission 18**, Artin!

In Missions 1–17 you learned individual capabilities: Python, JSON team data, deterministic coaching tools, computer vision, player tracking, persistent memory, RAG, and agentic decision-making.

Now we integrate those pieces into one real product: **The AI Soccer Agent**.

The central idea is:

```text
Player + Team Data
        +
Memory
        +
RAG Knowledge
        +
Match Tracking
        +
Coach Tools
        +
Agent Decisions
        +
Gemini
        ↓
AI Soccer Coach
```


## 🎯 Mission Goals

By the end of this mission, you will build a multi-page Streamlit application that can:

1. Load the `Artin_FC_v3.json` team database.
2. Select players from the application.
3. Display player statistics and Plotly charts.
4. Run deterministic coaching tools.
5. Store and recall persistent coaching memory.
6. Search tactical knowledge with RAG.
7. Upload a soccer video and track players.
8. Calculate movement, distance, speed, and tactical zones.
9. Combine player, team, memory, RAG, and match context.
10. Use an agentic decision layer to choose relevant tools.
11. Use Gemini to generate personalized coaching advice.


## 🏗️ Final Product Architecture

```text
ai_soccer_agent/
│
├── app.py
├── requirements.txt
│
├── data/
│   ├── Artin_FC_v3.json
│   ├── memory.json
│   └── tactical_docs.json
│
├── outputs/
│   ├── tracking/
│   ├── reports/
│   └── plots/
│
├── modules/
│   ├── coach_toolbox.py
│   ├── memory_toolbox.py
│   ├── knowledge_toolbox.py
│   ├── vision_tracker.py
│   ├── context_builder.py
│   └── agent_toolbox.py
│
└── pages/
    ├── 1_⚽_Roster_&_Stats.py
    ├── 2_🧠_Memory_Toolbox.py
    ├── 3_📹_Video_Tracking.py
    ├── 4_📚_Soccer_RAG.py
    └── 5_🤖_AI_Coach_Agent.py
```

Each module has one responsibility. The pages are mainly user interface; the reusable logic lives in `modules/`.


---
## 🗂️ Step 1 — Create the Project Scaffold


In [ ]:
import os

directories = [
    "ai_soccer_agent",
    "ai_soccer_agent/data",
    "ai_soccer_agent/outputs",
    "ai_soccer_agent/outputs/tracking",
    "ai_soccer_agent/outputs/reports",
    "ai_soccer_agent/outputs/plots",
    "ai_soccer_agent/modules",
    "ai_soccer_agent/pages",
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"✓ Directory confirmed: {directory}")

open("ai_soccer_agent/modules/__init__.py", "w", encoding="utf-8").close()
print("\nStep 1 Complete.")


---
## 📦 Step 2 — Define Application Dependencies

The vision stack uses Ultralytics YOLO + ByteTrack. Plotly is used for interactive charts. The NumPy upper bound is intentionally conservative because computer-vision packages can be sensitive to major NumPy changes.


In [ ]:
%%writefile ai_soccer_agent/requirements.txt
streamlit>=1.35,<2.0
google-genai>=1.0.0
opencv-python-headless>=4.8,<5.0
numpy>=1.24,<2.1
pandas>=2.0,<3.0
plotly>=5.18,<7.0
ultralytics>=8.3.0
mplsoccer>=1.4.0


Install from a terminal:

```bash
cd ai_soccer_agent
pip install -r requirements.txt
```


---
## 📄 Step 3 — Seed the Team, Memory, and RAG Data

`Artin_FC_v3.json` is the **team database**. It contains team identity, tactical identity, and player information. `memory.json` stores persistent coaching observations. `tactical_docs.json` is our first RAG knowledge base.


In [ ]:
import json

roster_data = {
    "team_name": "Artin FC",
    "team_profile": {
        "formation": "4-3-3",
        "playing_style": "Possession-based",
        "attack_direction": "Right",
        "tactics": {
            "build_up": "Build from the back",
            "pressing": "High press",
            "defensive_line": "Medium",
            "attacking_width": "Wide",
            "transition": "Fast"
        }
    },
    "players": [
        {
            "player_id": 1,
            "name": "Artin",
            "position": "Forward",
            "age": 15,
            "preferred_foot": "Right",
            "stats": {"shooting": 6.4, "stamina": 8.5, "passing": 8.1, "pace": 8.2, "dribbling": 7.8, "defense": 5.0}
        },
        {
            "player_id": 2,
            "name": "Leo",
            "position": "Midfielder",
            "age": 16,
            "preferred_foot": "Right",
            "stats": {"shooting": 7.8, "stamina": 5.2, "passing": 8.8, "pace": 7.5, "dribbling": 8.0, "defense": 6.5}
        },
        {
            "player_id": 3,
            "name": "Kian",
            "position": "Defender",
            "age": 15,
            "preferred_foot": "Left",
            "stats": {"shooting": 4.5, "stamina": 9.0, "passing": 6.8, "pace": 7.0, "dribbling": 6.0, "defense": 8.8}
        }
    ]
}

memory_data = {
    "Artin": [
        "Needs to work on first-time finishing inside the box.",
        "Demonstrates high stamina during full-pitch transitions."
    ],
    "Leo": [
        "Pace distribution requires management during second halves."
    ],
    "Kian": [
        "Strong defensive effort and stamina."
    ]
}

tactical_docs = [
    {"id": "doc_1", "topic": "Forward Movement", "position": "Forward", "text": "Forwards should create depth with diagonal runs into channels behind defenders. During defensive transitions, press immediate passing lanes."},
    {"id": "doc_2", "topic": "Midfield Positioning", "position": "Midfielder", "text": "Midfielders dictate tempo by maintaining half-turn body angles to receive passes forward. Scan space before receiving the ball."},
    {"id": "doc_3", "topic": "Defensive Positioning", "position": "Defender", "text": "Defenders should maintain compact distances between lines, protect central spaces, and adjust their position as the ball moves."},
    {"id": "doc_4", "topic": "Final Third", "position": "Forward", "text": "Forwards should occupy dangerous final-third spaces, stretch the defensive line, and make well-timed runs toward goal."},
    {"id": "doc_5", "topic": "Pressing", "position": "All", "text": "A coordinated press should close passing lanes, move as a unit, and apply pressure when the opponent receives with limited options."}
]

for filename, payload in [
    ("Artin_FC_v3.json", roster_data),
    ("memory.json", memory_data),
    ("tactical_docs.json", tactical_docs),
]:
    with open(f"ai_soccer_agent/data/{filename}", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=4, ensure_ascii=False)
    print(f"✓ {filename} created")


---
# Part 1 — Core Python Modules

The pages will call these modules instead of containing all the business logic themselves.


## 🧰 Step 4 — `coach_toolbox.py`

These functions are deterministic. If Python can answer a question reliably, we should not ask Gemini to invent the answer.


In [ ]:
%%writefile ai_soccer_agent/modules/coach_toolbox.py

def is_forward(player):
    return player.get("position", "").lower() == "forward"


def excellent_stamina(player):
    return player.get("stats", {}).get("stamina", 0) >= 7.5


def needs_shooting_practice(player):
    return player.get("stats", {}).get("shooting", 0) < 7.0


def needs_passing_practice(player):
    return player.get("stats", {}).get("passing", 0) < 7.0


def analyze_player(player):
    recommendations = []
    if needs_shooting_practice(player):
        recommendations.append("Shooting practice is recommended.")
    if needs_passing_practice(player):
        recommendations.append("Passing practice is recommended.")
    if excellent_stamina(player):
        recommendations.append("Stamina is a current strength.")
    if is_forward(player):
        recommendations.append("As a forward, focus on movement and positioning in the final third.")
    return recommendations


## 🧠 Step 5 — `memory_toolbox.py`

Memory provides persistence across sessions. It is separate from Gemini.


In [ ]:
%%writefile ai_soccer_agent/modules/memory_toolbox.py
import json
import os
from datetime import datetime

MEMORY_FILE = "data/memory.json"


def load_memory():
    if not os.path.exists(MEMORY_FILE):
        return {}
    with open(MEMORY_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def save_memory(memory):
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(memory, f, indent=4, ensure_ascii=False)


def get_player_memory(player_name):
    return load_memory().get(player_name, [])


def add_memory(player_name, observation):
    memory = load_memory()
    memory.setdefault(player_name, [])
    item = {
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "observation": observation
    }
    memory[player_name].append(item)
    save_memory(memory)
    return item


## 📚 Step 6 — `knowledge_toolbox.py`

This is the transparent first version of RAG. It retrieves the most relevant tactical documents using keyword overlap. The function interface can later be upgraded to embeddings/vector search without changing the AI agent.


In [ ]:
%%writefile ai_soccer_agent/modules/knowledge_toolbox.py
import json

KNOWLEDGE_FILE = "data/tactical_docs.json"


def load_knowledge():
    with open(KNOWLEDGE_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def retrieve_knowledge(question, top_k=3):
    documents = load_knowledge()
    words = {w.strip(".,!?;:").lower() for w in question.split() if len(w.strip(".,!?;:")) > 2}
    results = []
    for document in documents:
        searchable = " ".join([
            str(document.get("topic", "")),
            str(document.get("position", "")),
            str(document.get("text", ""))
        ]).lower()
        score = sum(1 for word in words if word in searchable)
        if score:
            results.append({"document": document, "score": score})
    results.sort(key=lambda item: item["score"], reverse=True)
    return results[:top_k]


## 👁️ Step 7 — `vision_tracker.py`

The vision module uses YOLO + ByteTrack and the bottom-center of each bounding box as the approximate player feet position.

It also includes:

- pixel → 120 × 80 pitch conversion
- frame-to-frame distance
- speed filtering
- attack-direction-aware zones

The pitch conversion is an **educational approximation** unless a real perspective calibration is added.


In [ ]:
%%writefile ai_soccer_agent/modules/vision_tracker.py
import math
import cv2
import numpy as np
import pandas as pd

PITCH_LENGTH = 120.0
PITCH_WIDTH = 80.0
MAX_REASONABLE_SPEED = 12.0


def convert_pixel_to_pitch(pixel_x, pixel_y, frame_width, frame_height):
    return (pixel_x / frame_width) * PITCH_LENGTH, (pixel_y / frame_height) * PITCH_WIDTH


def get_zone(pitch_x, attack_direction="Right"):
    if str(attack_direction).lower() == "left":
        if pitch_x < 40:
            return "Final Third"
        if pitch_x < 80:
            return "Midfield"
        return "Defense"
    if pitch_x < 40:
        return "Defense"
    if pitch_x < 80:
        return "Midfield"
    return "Final Third"


def calculate_movement_statistics(tracking_df):
    if tracking_df.empty:
        return {"distance_yards": 0.0, "distance_km": 0.0, "average_speed_yps": 0.0, "max_speed_yps": 0.0, "top_zone": "Unknown"}

    data = tracking_df.sort_values("frame").copy()
    distances, speeds, zones = [], [], []
    previous = None

    for _, row in data.iterrows():
        zones.append(row["zone"])
        if previous is not None:
            dx = row["pitch_x"] - previous["pitch_x"]
            dy = row["pitch_y"] - previous["pitch_y"]
            distance = math.sqrt(dx * dx + dy * dy)
            frame_difference = row["frame"] - previous["frame"]
            if frame_difference > 0 and row["fps"] > 0:
                speed = distance / (frame_difference / row["fps"])
                if speed <= MAX_REASONABLE_SPEED:
                    distances.append(distance)
                    speeds.append(speed)
        previous = row

    zone_counts = pd.Series(zones).value_counts()
    return {
        "distance_yards": round(sum(distances), 2),
        "distance_km": round(sum(distances) * 0.0009144, 3),
        "average_speed_yps": round(float(np.mean(speeds)), 2) if speeds else 0.0,
        "max_speed_yps": round(float(np.max(speeds)), 2) if speeds else 0.0,
        "top_zone": zone_counts.index[0] if not zone_counts.empty else "Unknown"
    }


def track_video(video_path, selected_tracking_id=None, attack_direction="Right", model_name="yolov8s.pt", max_frames=None):
    from ultralytics import YOLO

    model = YOLO(model_name)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    rows = []
    frame_number = 0

    while True:
        if max_frames is not None and frame_number >= max_frames:
            break
        success, frame = cap.read()
        if not success:
            break

        results = model.track(frame, persist=True, tracker="bytetrack.yaml", classes=[0], conf=0.35, imgsz=1280, verbose=False)
        result = results[0]

        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.cpu().numpy().astype(int)
            for box, tracking_id in zip(boxes, ids):
                x1, y1, x2, y2 = box
                pixel_x = (x1 + x2) / 2
                pixel_y = y2
                pitch_x, pitch_y = convert_pixel_to_pitch(pixel_x, pixel_y, width, height)
                rows.append({
                    "frame": frame_number,
                    "tracking_id": int(tracking_id),
                    "pixel_x": float(pixel_x),
                    "pixel_y": float(pixel_y),
                    "pitch_x": float(pitch_x),
                    "pitch_y": float(pitch_y),
                    "fps": float(fps),
                    "zone": get_zone(pitch_x, attack_direction)
                })
        frame_number += 1

    cap.release()
    tracking_df = pd.DataFrame(rows)
    if selected_tracking_id is not None and not tracking_df.empty:
        tracking_df = tracking_df[tracking_df["tracking_id"] == int(selected_tracking_id)].copy()
    return tracking_df, {"fps": fps, "width": width, "height": height, "frames_processed": frame_number}


## 🧩 Step 8 — `context_builder.py`

The context builder creates one unified package of information for the agent.


In [ ]:
%%writefile ai_soccer_agent/modules/context_builder.py
import json
from modules.memory_toolbox import get_player_memory
from modules.knowledge_toolbox import retrieve_knowledge

TEAM_FILE = "data/Artin_FC_v3.json"


def load_team_data():
    with open(TEAM_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def get_players():
    return load_team_data().get("players", [])


def get_player(player_name):
    for player in get_players():
        if player.get("name") == player_name:
            return player
    return None


def get_team_profile():
    return load_team_data().get("team_profile", {})


def get_coach_context(player_name, question="", match_data=None):
    player = get_player(player_name)
    if player is None:
        raise ValueError(f"Player '{player_name}' was not found.")
    data = load_team_data()
    return {
        "team": {"name": data.get("team_name", "Unknown Team"), "profile": data.get("team_profile", {})},
        "player": player,
        "memory": get_player_memory(player_name),
        "knowledge": retrieve_knowledge(question),
        "match": match_data or {}
    }


## 🤖 Step 9 — `agent_toolbox.py`

This is the Mission 17 → Mission 18 bridge.

The agent decides which tools and information are relevant, executes deterministic tools, builds a grounded prompt, and then asks Gemini to reason over the collected facts.


In [ ]:
%%writefile ai_soccer_agent/modules/agent_toolbox.py
import json
import os

from modules.coach_toolbox import analyze_player, needs_shooting_practice, needs_passing_practice
from modules.context_builder import get_coach_context


def get_coach_decisions(question, context):
    q = question.lower()
    decisions = []
    if any(x in q for x in ["shoot", "finishing", "score", "scoring"]):
        decisions.append({"tool": "needs_shooting_practice", "reason": "The question concerns finishing."})
    if any(x in q for x in ["pass", "passing"]):
        decisions.append({"tool": "needs_passing_practice", "reason": "The question concerns passing."})
    if any(x in q for x in ["run", "movement", "speed", "distance", "tracking", "match"]):
        decisions.append({"tool": "match_analysis", "reason": "The question concerns match movement or workload."})
    if any(x in q for x in ["tactic", "position", "forward", "defend", "press", "final third"]):
        decisions.append({"tool": "retrieve_knowledge", "reason": "The question requires tactical knowledge."})
    if not decisions:
        decisions.append({"tool": "general_player_analysis", "reason": "Use player context for a general coaching answer."})
    return decisions


def execute_coach_tools(player, decisions, context):
    results = {}
    for decision in decisions:
        name = decision["tool"]
        if name == "needs_shooting_practice":
            results[name] = needs_shooting_practice(player)
        elif name == "needs_passing_practice":
            results[name] = needs_passing_practice(player)
        elif name == "general_player_analysis":
            results[name] = analyze_player(player)
        elif name == "match_analysis":
            results[name] = context.get("match", {})
        elif name == "retrieve_knowledge":
            results[name] = context.get("knowledge", [])
    return results


def format_knowledge(knowledge):
    output = []
    for item in knowledge:
        doc = item.get("document", {})
        output.append({"topic": doc.get("topic"), "position": doc.get("position"), "text": doc.get("text"), "score": item.get("score")})
    return output


def build_prompt(question, context, decisions, tool_results):
    grounded = {
        "team": context.get("team"),
        "player": context.get("player"),
        "memory": context.get("memory"),
        "knowledge": format_knowledge(context.get("knowledge", [])),
        "match": context.get("match", {}),
        "decisions": decisions,
        "tool_results": tool_results
    }
    instructions = [
        "You are an AI Soccer Coach for Artin FC.",
        "Give accurate, personalized, supportive coaching advice.",
        "Do not invent player statistics.",
        "Treat Python-calculated values as facts.",
        "Use retrieved tactical knowledge when relevant.",
        "Use memory to maintain continuity.",
        "If match data is unavailable, say so.",
        "Separate measured facts from coaching interpretation.",
        "Give practical advice appropriate for the player's position."
    ]
    return "\n".join(instructions) + "\n\nCONTEXT:\n" + json.dumps(grounded, indent=2, ensure_ascii=False) + "\n\nPLAYER QUESTION:\n" + question + "\n\nStructure the answer as: 1. What the data tells us 2. Why it matters tactically 3. What the player should do next"


def generate_gemini_response(question, player_name, match_data=None, model_name="gemini-2.5-flash"):
    from google import genai
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("GEMINI_API_KEY is not set.")

    context = get_coach_context(player_name, question, match_data=match_data)
    decisions = get_coach_decisions(question, context)
    tool_results = execute_coach_tools(context["player"], decisions, context)
    prompt = build_prompt(question, context, decisions, tool_results)

    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(model=model_name, contents=prompt)

    return {"answer": response.text, "decisions": decisions, "tool_results": tool_results, "context": context}


---
# Part 2 — Build the Streamlit Application

Streamlit automatically discovers Python files in the `pages/` folder. Each page uses the same modules and Streamlit session state for the currently selected player.


## 🏠 Step 10 — `app.py`

In [ ]:
%%writefile ai_soccer_agent/app.py
import streamlit as st

st.set_page_config(page_title="AI Soccer Agent", page_icon="⚽", layout="wide")

st.title("⚽ AI Soccer Agent")
st.subheader("Mission 18 — Capstone Product")
st.markdown("""
Welcome to the **AI Soccer Agent**.

This application combines:

- ⚽ Team and player data
- 🧰 Deterministic coaching tools
- 🧠 Persistent memory
- 📚 Soccer RAG
- 📹 Player tracking
- 🤖 Agentic decision-making
- ✨ Gemini reasoning

Use the navigation on the left to explore the system.
""")
st.info("Start with **Roster & Stats**, then explore Memory, Video Tracking, Soccer RAG, and finally the AI Coach.")


## ⚽ Step 11 — Roster & Stats Page

In [ ]:
%%writefile ai_soccer_agent/pages/1_⚽_Roster_&_Stats.py
import streamlit as st
import plotly.graph_objects as go
from modules.context_builder import load_team_data, get_players
from modules.coach_toolbox import analyze_player

st.title("⚽ Roster & Player Stats")
data = load_team_data()
profile = data.get("team_profile", {})

c1, c2, c3 = st.columns(3)
c1.metric("Formation", profile.get("formation", "Unknown"))
c2.metric("Style", profile.get("playing_style", "Unknown"))
c3.metric("Attack Direction", profile.get("attack_direction", "Unknown"))

players = get_players()
names = [p["name"] for p in players]
selected = st.selectbox("Select Player", names)
st.session_state["selected_player"] = selected
player = next(p for p in players if p["name"] == selected)

st.subheader(f"Player Profile — {selected}")
st.write("**Position:**", player.get("position"))
st.write("**Age:**", player.get("age"))
st.write("**Preferred Foot:**", player.get("preferred_foot"))

stats = player.get("stats", {})
fig = go.Figure(go.Bar(x=list(stats.keys()), y=list(stats.values())))
fig.update_layout(title="Player Performance", yaxis_title="Rating", yaxis_range=[0, 10])
st.plotly_chart(fig, use_container_width=True)

st.subheader("🧰 Coach Toolbox")
for recommendation in analyze_player(player):
    st.write("•", recommendation)


## 🧠 Step 12 — Memory Toolbox Page

In [ ]:
%%writefile ai_soccer_agent/pages/2_🧠_Memory_Toolbox.py
import streamlit as st
from modules.context_builder import get_players
from modules.memory_toolbox import get_player_memory, add_memory

st.title("🧠 Memory Toolbox")
players = get_players()
names = [p["name"] for p in players]
default = st.session_state.get("selected_player", names[0])
if default not in names:
    default = names[0]
player = st.selectbox("Player", names, index=names.index(default))
st.session_state["selected_player"] = player

st.subheader("Stored Memories")
memories = get_player_memory(player)
if not memories:
    st.info("No memories stored for this player yet.")
else:
    for i, item in enumerate(memories, 1):
        if isinstance(item, dict):
            st.write(f"**{i}. {item.get('date', '')}** — {item.get('observation', '')}")
        else:
            st.write(f"**{i}.** {item}")

st.divider()
st.subheader("Add Coaching Memory")
observation = st.text_area("Observation", placeholder="Example: Player improved movement into the final third.")
if st.button("Save Memory"):
    if observation.strip():
        saved = add_memory(player, observation.strip())
        st.success("Memory saved.")
        st.json(saved)
    else:
        st.warning("Please enter an observation.")


## 📹 Step 13 — Video Tracking Page

Start with a short clip. YOLO inference can be computationally expensive. The page saves the selected player's tracking CSV and puts the latest movement statistics into `st.session_state` for the AI Coach page.


In [ ]:
%%writefile ai_soccer_agent/pages/3_📹_Video_Tracking.py
import os
import tempfile
import streamlit as st
from modules.context_builder import get_team_profile
from modules.vision_tracker import track_video, calculate_movement_statistics

st.title("📹 Video Tracking")
video = st.file_uploader("Upload a soccer video", type=["mp4", "mov", "avi"])
if video is None:
    st.info("Upload a short soccer video to begin.")
    st.stop()

profile = get_team_profile()
attack_default = profile.get("attack_direction", "Right")
attack_direction = st.selectbox("Attack Direction", ["Right", "Left"], index=0 if attack_default == "Right" else 1)
max_frames = st.number_input("Maximum frames to process", min_value=30, max_value=3000, value=300, step=30)

if st.button("🔎 Detect and Track Players"):
    suffix = os.path.splitext(video.name)[1]
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(video.getbuffer())
        video_path = tmp.name
    try:
        with st.spinner("Running YOLO + ByteTrack..."):
            df, metadata = track_video(video_path, attack_direction=attack_direction, max_frames=int(max_frames))
        if df.empty:
            st.warning("No tracked players were found.")
            st.stop()
        st.success(f"Processed {metadata['frames_processed']} frames.")
        ids = sorted(df["tracking_id"].unique().tolist())
        selected_id = st.selectbox("Select Tracking ID", ids)
        player_df = df[df["tracking_id"] == selected_id].copy()
        stats = calculate_movement_statistics(player_df)

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Distance", f"{stats['distance_yards']:.1f} yd")
        c2.metric("Distance", f"{stats['distance_km']:.3f} km")
        c3.metric("Avg Speed", f"{stats['average_speed_yps']:.2f} yd/s")
        c4.metric("Top Zone", stats["top_zone"])

        st.subheader("Movement Data")
        st.dataframe(player_df, use_container_width=True)
        st.subheader("Movement Trajectory")
        st.line_chart(player_df.sort_values("frame").set_index("frame")[["pitch_x", "pitch_y"]])

        output = f"outputs/tracking/tracking_id_{selected_id}.csv"
        player_df.to_csv(output, index=False)
        st.success(f"Saved: `{output}`")
        st.session_state["latest_match_data"] = stats
    except Exception as error:
        st.error(f"Tracking failed: {error}")
    finally:
        if os.path.exists(video_path):
            os.remove(video_path)


## 📚 Step 14 — Soccer RAG Page

In [ ]:
%%writefile ai_soccer_agent/pages/4_📚_Soccer_RAG.py
import streamlit as st
from modules.knowledge_toolbox import retrieve_knowledge

st.title("📚 Soccer RAG")
st.write("Search the tactical knowledge base before asking the AI Coach.")
question = st.text_input("Ask a soccer knowledge question", placeholder="What should a forward do in the final third?")

if st.button("🔎 Search Knowledge"):
    if not question.strip():
        st.warning("Enter a question first.")
        st.stop()
    results = retrieve_knowledge(question)
    if not results:
        st.info("No matching documents were found.")
    for i, result in enumerate(results, 1):
        doc = result["document"]
        with st.expander(f"{i}. {doc.get('topic')} (score: {result['score']})"):
            st.write("**Position:**", doc.get("position"))
            st.write(doc.get("text"))


## 🤖 Step 15 — AI Coach Agent Page

In [ ]:
%%writefile ai_soccer_agent/pages/5_🤖_AI_Coach_Agent.py
import streamlit as st
from modules.context_builder import get_players
from modules.agent_toolbox import generate_gemini_response

st.title("🤖 AI Soccer Coach")
players = get_players()
names = [p["name"] for p in players]
default = st.session_state.get("selected_player", names[0])
if default not in names:
    default = names[0]
player = st.selectbox("Player", names, index=names.index(default))
st.session_state["selected_player"] = player

question = st.text_area("Ask your AI Coach", placeholder="Why should I improve my movement in the final third?", height=120)
match_data = st.session_state.get("latest_match_data", {})
if match_data:
    st.info("The latest video-tracking statistics are available to the agent.")
else:
    st.info("No match-tracking data is available yet. Player data, memory, and RAG can still be used.")

if st.button("⚽ Ask Coach"):
    if not question.strip():
        st.warning("Please enter a question.")
        st.stop()
    try:
        with st.spinner("The AI Coach is thinking..."):
            result = generate_gemini_response(question, player, match_data=match_data)
        st.subheader("💬 Coach Response")
        st.write(result["answer"])
        with st.expander("🔧 Agent Decisions"):
            st.json(result["decisions"])
        with st.expander("🧰 Tool Results"):
            st.json(result["tool_results"])
    except Exception as error:
        st.error(f"AI Coach error: {error}")
        st.info("Check that GEMINI_API_KEY is configured and google-genai is installed.")


---
# Part 3 — Test the Components

Before launching the full application, test the modules independently.


In [ ]:
import os, sys
project_root = os.path.abspath("ai_soccer_agent")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.context_builder import get_players, get_player, get_team_profile
from modules.coach_toolbox import analyze_player
from modules.memory_toolbox import get_player_memory
from modules.knowledge_toolbox import retrieve_knowledge

players = get_players()
print("Players:", [p["name"] for p in players])
print("Team profile:", get_team_profile())

artin = get_player("Artin")
print("\nArtin:", artin)
print("\nCoach analysis:")
for item in analyze_player(artin):
    print("-", item)

print("\nMemory:", get_player_memory("Artin"))
print("\nRAG results:")
for result in retrieve_knowledge("What should a forward do in the final third?"):
    print(result["document"]["topic"], "score=", result["score"])


---
# 🔐 Gemini Configuration

The AI Coach uses the Google GenAI SDK. Keep the API key outside your source code.

### macOS / Linux

```bash
export GEMINI_API_KEY="YOUR_API_KEY"
```

### Windows PowerShell

```powershell
$env:GEMINI_API_KEY="YOUR_API_KEY"
```

The application reads the key with:

```python
os.getenv("GEMINI_API_KEY")
```


---
# ▶️ Launch the Product

From a terminal:

```bash
cd ai_soccer_agent
pip install -r requirements.txt
streamlit run app.py
```

### Recommended testing order

1. ⚽ Roster & Stats
2. 🧠 Memory Toolbox
3. 📚 Soccer RAG
4. 📹 Video Tracking
5. 🤖 AI Coach Agent

Test the lower-level systems before testing the complete agent.


---
# 🧪 Final Boss Challenge

Select **Artin**, analyze a short match video, and then ask:

> **"What should I improve based on my player profile, my match movement, and what you remember about me?"**

A strong answer should contain:

1. At least one fact from `Artin_FC_v3.json`.
2. At least one measured match fact when tracking is available.
3. At least one relevant memory.
4. At least one tactical principle from RAG.
5. A practical recommendation.

The agent should not invent statistics.


# 🧠 The Most Important Lesson

Gemini is **not the whole AI agent**.

```text
AI Agent
=
LLM
+
Tools
+
Memory
+
RAG
+
Player Data
+
Team Data
+
Match Data
+
Decision Logic
+
Application UI
```

The LLM provides generative reasoning.

Python tools calculate deterministic facts.

Memory provides continuity.

RAG provides domain knowledge.

The vision system provides observations from video.

The agent decides which information and tools are relevant.

Together, these components form the AI Soccer Agent.


# 🎓 Reflection Questions

Answer these in your own words:

1. What is the difference between a chatbot and an AI agent?
2. Why should Python calculate deterministic statistics instead of Gemini?
3. Why does the coach need memory?
4. What problem does RAG solve?
5. What information does the video tracker provide?
6. What does `context_builder.py` do?
7. Why are Streamlit pages separated from reusable modules?
8. What happens from the moment a player asks a question until the coach responds?
9. Which parts of the system are deterministic?
10. Which part performs generative reasoning?

### Final question

> **If Gemini disappeared tomorrow, which parts of your AI Soccer Agent would still work?**


# 🏁 Mission 18 Complete

You have moved from individual Python exercises to a complete AI application.

The final flow is:

```text
observe → retrieve → remember → decide → reason → coach
```

That is the foundation of a real AI agent product.
